In [ ]:
import pandas as pd
from sklearn.metrics import f1_score

In [ ]:
columns = [
    "research_type", 
    "affiliation", 
    "pseudocode", 
    "open_source_code", 
    "train", 
    "validation",  
    "hardware_specification", 
    "software_dependencies", 
    "experiment_setup"
]

In [ ]:
manual_eval_file = "../../test_set/manual_eval.csv"
llm_eval_file = "../../test_set/llm_eval.csv"

In [ ]:
# Calulate the f1 score for the columns

# Read the manual evaluation data
manual_eval = pd.read_csv(manual_eval_file)
# Read the LLM evaluation data
llm_eval = pd.read_csv(llm_eval_file)

# Initialize a dictionary to store the f1 scores
f1_scores = {}

# Sort manual_eval and llm_eval by the filename column
manual_eval = manual_eval.sort_values(by="filename").reset_index(drop=True)
llm_eval = llm_eval.sort_values(by="filename").reset_index(drop=True)

# Calculate the f1 score for each column
for column in columns:
    f1 = f1_score(manual_eval[column], llm_eval[column])
    f1_scores[column] = f1


In [ ]:
for key, value in f1_scores.items():
    print(f"F1 score for {key}: {value * 100:.2f}")

In [ ]:
# Visualize a confusion matrix for each column
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix
# put them all on one plot 2 rows and 5 columns
fig, axes = plt.subplots(2, 5, figsize=(20, 10))
for i, column in enumerate(columns):

    cm = confusion_matrix(manual_eval[column], llm_eval[column])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=[0, 1], yticklabels=[0, 1], ax=axes[i//5, i%5])
    axes[i//5, i%5].set_xlabel('LLM Prediction')
    axes[i//5, i%5].set_ylabel('Manual Evaluation')
    axes[i//5, i%5].set_title(f'Confusion Matrix for {column}')
    # just display the matrix not the color bar
    axes[i//5, i%5].collections[0].colorbar.remove()

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

results_eval_file = "../../results/llm_results.csv"
results_eval = pd.read_csv(results_eval_file)

conference_colors = {
    'AAAI': '#f3b23e',
    'ICLR': '#e37137',
    'ICML': '#d62728',
    'IJCAI': '#e662bd',
    'NeurIPS': '#5dbaf7',
}

manual_conference_counts = manual_eval['conference'].value_counts().sort_index()
results_conference_counts = results_eval['conference'].value_counts().sort_index()
manual_conference_pie_colors = [conference_colors.get(conference, '#cccccc') for conference in manual_conference_counts.index]
results_conference_pie_colors = [conference_colors.get(conference, '#cccccc') for conference in results_conference_counts.index]

fig, axes = plt.subplots(1, 2, figsize=(16, 8))

axes[0].pie(manual_conference_counts, labels=manual_conference_counts.index, colors=manual_conference_pie_colors, autopct='%1.1f%%', startangle=140, textprops={'fontsize': 13})
axes[0].set_title('Distribution of Conferences in Validation Set (160 papers)', fontsize=18)
axes[0].axis('equal')

axes[1].pie(results_conference_counts, labels=results_conference_counts.index, colors=results_conference_pie_colors, autopct='%1.1f%%', startangle=140, textprops={'fontsize': 13})
axes[1].set_title('Distribution of Conferences in All LLM Results (56800 papers)', fontsize=18)
axes[1].axis('equal')

plt.tight_layout()
plt.savefig(f"../../plots/test_set_conference_distribution.pdf", dpi=600, bbox_inches='tight')
plt.show()

In [ ]:
import matplotlib.pyplot as plt

results_eval_file = "../../results/llm_results.csv"
results_eval = pd.read_csv(results_eval_file)

year_palette = [
    '#f6c453',
    '#f3b23e',
    '#ef9f3a',
    '#e88939',
    '#e37137',
    '#dc5b4b',
    '#d62728',
    '#df4f83',
    '#e662bd',
    '#9c7ae6',
    '#5dbaf7',
]

manual_year_counts = manual_eval['year'].value_counts().sort_index()
results_year_counts = results_eval['year'].value_counts().sort_index()
year_labels = sorted(set(manual_year_counts.index).union(results_year_counts.index))
year_colors = {year: year_palette[i % len(year_palette)] for i, year in enumerate(year_labels)}
manual_year_pie_colors = [year_colors[year] for year in manual_year_counts.index]
results_year_pie_colors = [year_colors[year] for year in results_year_counts.index]

fig, axes = plt.subplots(1, 2, figsize=(16, 8))

axes[0].pie(manual_year_counts, labels=manual_year_counts.index, colors=manual_year_pie_colors, autopct='%1.1f%%', startangle=140, textprops={'fontsize': 13})
axes[0].set_title('Distribution of Years in Validation Set (160 papers)', fontsize=18)
axes[0].axis('equal')

axes[1].pie(results_year_counts, labels=results_year_counts.index, colors=results_year_pie_colors, autopct='%1.1f%%', startangle=140, textprops={'fontsize': 13})
axes[1].set_title('Distribution of Years in All LLM Results (56800 papers)', fontsize=18)
axes[1].axis('equal')

plt.tight_layout()
plt.savefig(f"../../plots/test_set_year_distribution.pdf", dpi=600, bbox_inches='tight')

plt.show()

In [ ]:
conference_labels = sorted(set(manual_eval['conference']).union(results_eval['conference']))
manual_conference_percent = manual_eval['conference'].value_counts(normalize=True).mul(100)
results_conference_percent = results_eval['conference'].value_counts(normalize=True).mul(100)

conference_percentage_table = pd.DataFrame({
    'conference': conference_labels,
    '% evaluation dataset': [manual_conference_percent.get(conference, 0) for conference in conference_labels],
    '% all llm results': [results_conference_percent.get(conference, 0) for conference in conference_labels],
})

conference_percentage_table['% evaluation dataset'] = conference_percentage_table['% evaluation dataset'].map(lambda value: f'{value:.1f}%')
conference_percentage_table['% all llm results'] = conference_percentage_table['% all llm results'].map(lambda value: f'{value:.1f}%')

conference_percentage_table

In [ ]:
year_labels = sorted(set(manual_eval['year']).union(results_eval['year']))
manual_year_percent = manual_eval['year'].value_counts(normalize=True).mul(100)
results_year_percent = results_eval['year'].value_counts(normalize=True).mul(100)

year_percentage_table = pd.DataFrame({
    'year': year_labels,
    '% evaluation dataset': [manual_year_percent.get(year, 0) for year in year_labels],
    '% all llm results': [results_year_percent.get(year, 0) for year in year_labels],
})

year_percentage_table['% evaluation dataset'] = year_percentage_table['% evaluation dataset'].map(lambda value: f'{value:.1f}%')
year_percentage_table['% all llm results'] = year_percentage_table['% all llm results'].map(lambda value: f'{value:.1f}%')

year_percentage_table